In [2]:
import pandas as pd
import json

# -----------------------------
# STEP 1: Load datasets
# -----------------------------
restaurants = pd.read_csv("new_orleans_restaurants.csv")

# Load business JSON (for attributes like price, takeout, etc.)
business_data = []
with open("yelp_academic_dataset_business.json", "r") as f:
    for line in f:
        business_data.append(json.loads(line))

business_df = pd.DataFrame(business_data)

In [3]:
# -----------------------------
# STEP 2: Filter business data to only restaurants in your dataset
# -----------------------------
business_df = business_df[business_df["business_id"].isin(restaurants["business_id"])]

In [4]:
# -----------------------------
# STEP 3: Extract attributes safely
# -----------------------------
def get_attribute(attr_dict, key):
    if isinstance(attr_dict, dict):
        return attr_dict.get(key)
    return None

# Extract price range
business_df["price_range"] = business_df["attributes"].apply(
    lambda x: get_attribute(x, "RestaurantsPriceRange2")
)

# Extract binary features
features = {
    "takeout": "RestaurantsTakeOut",
    "delivery": "RestaurantsDelivery",
    "reservations": "RestaurantsReservations",
    "outdoor_seating": "OutdoorSeating",
    "good_for_kids": "GoodForKids"
}

for new_col, attr_name in features.items():
    business_df[new_col] = business_df["attributes"].apply(
        lambda x: get_attribute(x, attr_name)
    )

In [5]:
# -----------------------------
# STEP 4: Clean binary features
# -----------------------------
def convert_to_binary(val):
    if val in [True, "True", "true", 1]:
        return 1
    else:
        return 0

for col in features.keys():
    business_df[col] = business_df[col].apply(convert_to_binary)

# Convert price to numeric
business_df["price_range"] = pd.to_numeric(business_df["price_range"], errors="coerce")


In [6]:
# -----------------------------
# STEP 5: Merge with restaurants CSV
# -----------------------------
merged_df = restaurants.merge(
    business_df[
        ["business_id", "price_range"] + list(features.keys())
    ],
    on="business_id",
    how="left"
)

In [7]:
# -----------------------------
# STEP 6: Build competition_density
# -----------------------------
competition = merged_df.groupby("postal_code").size().reset_index(name="competition_density")

merged_df = merged_df.merge(competition, on="postal_code", how="left")


In [8]:
# -----------------------------
# STEP 7: Handle missing values
# -----------------------------
# Fill price with median
merged_df["price_range"] = merged_df["price_range"].fillna(merged_df["price_range"].median())

# Fill binary features with 0
for col in features.keys():
    merged_df[col] = merged_df[col].fillna(0)


In [10]:
# -----------------------------
# STEP X: Create Business Age (from CSV)
# -----------------------------

# Load review CSV
review_df = pd.read_csv("new_orleans_reviews.csv")

# Keep only relevant businesses (extra safety, optional)
review_df = review_df[review_df["business_id"].isin(restaurants["business_id"])]

# Convert date column to datetime
review_df["date"] = pd.to_datetime(review_df["date"])

# Get earliest review date per business
first_review = review_df.groupby("business_id")["date"].min().reset_index()
first_review.rename(columns={"date": "first_review_date"}, inplace=True)

# Define "today"
today = pd.to_datetime("2026-04-24")

# Calculate business age in days
first_review["business_age_days"] = (today - first_review["first_review_date"]).dt.days

# Merge into main dataset
merged_df = merged_df.merge(
    first_review[["business_id", "business_age_days"]],
    on="business_id",
    how="left"
)

# Fill missing values (if any businesses have no reviews)
merged_df["business_age_days"] = merged_df["business_age_days"].fillna(0)

In [11]:
# -----------------------------
# STEP 8: Final feature table
# -----------------------------
feature_table = merged_df[
    [
        "business_id",
        "stars",
        "review_count",
        "price_range",
        "takeout",
        "delivery",
        "reservations",
        "outdoor_seating",
        "good_for_kids",
        "postal_code",
        "competition_density",
        "business_age_days"   # 👈 ADD THIS
    ]
]
# Fill missing ages (if any)
merged_df["business_age_days"] = merged_df["business_age_days"].fillna(0)

In [12]:
# -----------------------------
# STEP 9: Save output
# -----------------------------
feature_table.to_csv("new_feature_table.csv", index=False)

print("✅ Feature table created: feature_table.csv")
print(feature_table.head())

✅ Feature table created: feature_table.csv
              business_id  stars  review_count  price_range  takeout  \
0  YNjyv0gfOr2g8lbmUpTnKg    4.5           350          2.0      1.0   
1  TLZ3-eDPLhUzfsWO4ad6Ug    4.0           382          2.0      1.0   
2  FRYkg_JvsWU9xIXZsEZcVA    3.5            27          3.0      0.0   
3  4IcB3QyMEA85UTWFKh9O9A    4.5             8          1.0      0.0   
4  Edg22x3CZkIv0GUib2oEFA    3.5           149          2.0      1.0   

   delivery  reservations  outdoor_seating  good_for_kids  postal_code  \
0       1.0           1.0              1.0            0.0      70112.0   
1       0.0           0.0              0.0            1.0      70112.0   
2       0.0           1.0              1.0            0.0      70115.0   
3       0.0           0.0              0.0            0.0      70119.0   
4       1.0           1.0              1.0            1.0      70118.0   

   competition_density  business_age_days  
0                173.0             